# 02. 벡터 인덱스 구축

24개 Document를 청크로 분할 → Upstage Embedding으로 벡터화 → FAISS 로컬 인덱스 저장.

**진행 순서**
1. 환경 설정 (UPSTAGE_API_KEY 로드)
2. `data/raw/*.md` 24개 로드 → `documents`
3. 청크 분할 → `chunks` (목표 50~110개)
4. 메타데이터 전파 검증
5. Embedding + FAISS 인덱스 (저장: `../data/faiss_index/`)
6. 검색 동작 검증 (시연 질문 1·2번 미리 테스트)


## 1. 환경 설정

In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
api_key = os.getenv("UPSTAGE_API_KEY")
if api_key:
    print(f"API Key 로드 성공: {api_key[:8]}...")
else:
    raise ValueError(
        "UPSTAGE_API_KEY를 찾을 수 없습니다.\n"
        "프로젝트 루트의 .env에 UPSTAGE_API_KEY=sk-...를 입력하세요."
    )


API Key 로드 성공: up_cCu2s...


## 2. data/raw/*.md 24개 로드

01_collect_data.ipynb에서 저장한 .md 파일을 다시 읽어 Document 리스트로 변환.
YAML frontmatter는 metadata로, 본문은 page_content로 분리한다.

**회귀 검증**: 24개 / 81,218자 (Cell 7·8 결과와 일치해야 함)


In [2]:
from pathlib import Path
import yaml
from langchain_core.documents import Document

RAW_DIR = Path("../data/raw")

def load_md_with_frontmatter(fpath):
    """YAML frontmatter가 있는 .md 파일을 Document로 변환."""
    text = fpath.read_text(encoding="utf-8")
    if not text.startswith("---\n"):
        raise ValueError(f"frontmatter 없음: {fpath.name}")
    _, fm, body = text.split("---\n", 2)
    metadata = yaml.safe_load(fm)
    return Document(page_content=body.strip(), metadata=metadata)

documents = [load_md_with_frontmatter(fp) for fp in sorted(RAW_DIR.glob("*.md"))]

# 회귀 검증 (Cell 7·8 결과와 일치 확인)
total_chars = sum(len(d.page_content) for d in documents)
assert len(documents) == 25, f"문서 수 불일치: {len(documents)} (기대 25)"
assert total_chars == 83433, f"본문 자수 불일치: {total_chars} (기대 83433)"
print(f"✓ 로드 완료: {len(documents)}개 / {total_chars:,}자 (Cell 7·8과 일치)")

# 샘플 메타데이터 확인
print(f"\n[샘플] {documents[0].metadata.get('title')}")
print(f"  metadata keys: {sorted(documents[0].metadata.keys())}")


✓ 로드 완료: 25개 / 83,433자 (Cell 7·8과 일치)

[샘플] Guam Wikivoyage - Do
  metadata keys: ['category', 'section', 'source', 'title']


## 3. 청크 분할 (RecursiveCharacterTextSplitter)

- `chunk_size=1200` — 강의 S2-1 가이드(법률 단락 800~1200) 안. 800·1000·1200 실측 비교 후 채택
- `chunk_overlap=150` — 단락 경계에서 맥락 유지
- `separators=["\n\n", "\n", " ", ""]` — 단락 → 줄 → 공백 → 글자 순으로 분할 시도

**목표 청크 수: 50~110개**. 실측 90개로 범위 안.


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=150,
    separators=["\n\n", "\n", " ", ""],
)
chunks = splitter.split_documents(documents)

# 통계
n = len(chunks)
avg_len = sum(len(c.page_content) for c in chunks) // n
max_len = max(len(c.page_content) for c in chunks)
min_len = min(len(c.page_content) for c in chunks)

print(f"원본 {len(documents)}개 Document → {n}개 청크")
print(f"  평균 길이: {avg_len}자")
print(f"  최대 길이: {max_len}자")
print(f"  최소 길이: {min_len}자")

# 청크 수가 50~110 범위인지
if 50 <= n <= 110:
    print(f"\n✓ 청크 수 {n}개 — 목표 범위(50~110) 안")
else:
    print(f"\n⚠️  청크 수 {n}개가 목표 범위(50~110) 밖. chunk_size 조정 검토 필요.")


원본 25개 Document → 93개 청크
  평균 길이: 906자
  최대 길이: 1198자
  최소 길이: 121자

✓ 청크 수 93개 — 목표 범위(50~110) 안


## 4. 메타데이터 전파 검증

`split_documents()`는 원본 Document의 metadata를 *모든* 청크에 복사한다.
각 청크에 `source`, `category`, `title`이 있고,
Wikivoyage 청크에는 `section`까지 정상 전파됐는지 확인.


In [4]:
from collections import Counter

# 모든 청크에 source/category/title 있는지
required_keys = {"source", "category", "title"}
for i, c in enumerate(chunks):
    missing = required_keys - set(c.metadata.keys())
    assert not missing, f"청크 {i} 메타 누락: {missing} ({c.metadata.get('title', '?')})"

# Wikivoyage 청크는 section 키도 있어야 함
wv_chunks = [c for c in chunks if "wikivoyage" in c.metadata.get("source", "").lower()]
for c in wv_chunks:
    assert "section" in c.metadata, f"Wikivoyage 청크에 section 없음: {c.metadata}"

print(f"✓ 모든 청크에 source/category/title 존재")
print(f"✓ Wikivoyage 청크 {len(wv_chunks)}개 모두 section 있음")

# 카테고리별 청크 수 분포
cat_count = Counter(c.metadata["category"] for c in chunks)
print(f"\n[카테고리별 청크 수]")
for cat, cnt in sorted(cat_count.items(), key=lambda x: -x[1]):
    print(f"  {cat:<10} {cnt:>3}개")


✓ 모든 청크에 source/category/title 존재
✓ Wikivoyage 청크 41개 모두 section 있음

[카테고리별 청크 수]
  general     68개
  교통          11개
  명소           4개
  리조트          4개
  안전·날씨        3개
  식당           2개
  액티비티         1개


## 5. Embedding + FAISS 인덱스

- **Embedding**: `UpstageEmbeddings(model="solar-embedding-1-large-query")` — 강사 솔루션·project_plan 표준
  - LangChain 0.7.7 `_invocation_params`이 자동으로 `-query` suffix를 떼고,
    `embed_documents()` 호출 시 `-passage`를 다시 붙이고,
    `embed_query()` 호출 시 `-query`를 다시 붙임 → 비대칭 검색 자동 보장
- **Vector Store**: FAISS (강의 표준)
- **저장 위치**: `../data/faiss_index/`

⚠️ 이 셀 실행 시 청크 개수만큼 Upstage API 호출 발생 (배치). 약 5~30초 소요.


In [5]:
from langchain_upstage import UpstageEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = UpstageEmbeddings(model="solar-embedding-1-large-query")

# FAISS 인덱스 구축 (chunks를 passage 모델로 임베딩)
vectorstore = FAISS.from_documents(chunks, embeddings)

# 로컬 저장
INDEX_DIR = "../data/faiss_index"
vectorstore.save_local(INDEX_DIR)

print(f"✓ FAISS 인덱스 구축 완료: {len(chunks)}개 벡터")
print(f"✓ 저장 위치: {Path(INDEX_DIR).resolve()}")

# 임베딩 차원 확인 (Upstage solar-embedding-1-large는 4096 차원)
sample_vec = embeddings.embed_query("test")
print(f"\n임베딩 차원: {len(sample_vec)}")


✓ FAISS 인덱스 구축 완료: 93개 벡터
✓ 저장 위치: C:\Users\wonwo\fastcampus\09.LLM\project\guam-family-chatbot\data\faiss_index

임베딩 차원: 4096


## 6. 검색 동작 검증

시연 질문 5개 중 RAG가 직접 답해야 하는 1·2번을 미리 테스트:

1. **"PIC 리조트 어때?"** → 리조트 카테고리에서 PIC 관련 청크가 top 3에 들어오는지
2. **"5월 둘째 주 괌 날씨"** → 안전·날씨 카테고리(특히 Climate 청크)가 top 3에 들어오는지

⚠️ 1번에서 PIC 관련 청크가 안 나오면 Sleep(535자) 본문에 PIC가 명시 안 된 것.
   2번에서 Climate 청크가 안 나오면 외교부 0404 보강 검토 트리거 ([E] 단계에서 결정).


In [6]:
# 검증 1: PIC 리조트
print("=" * 60)
print("Q1: PIC 리조트 어때?")
print("=" * 60)
results = vectorstore.similarity_search("PIC 리조트 어때?", k=3)
for i, doc in enumerate(results):
    cat = doc.metadata.get("category", "?")
    title = doc.metadata.get("title", "?")
    print(f"\n[{i+1}] [{cat}] {title}")
    print(f"    {doc.page_content[:120]}...")

pic_in_top3 = any("PIC" in doc.page_content for doc in results)
print(f"\n→ 'PIC' 키워드가 top 3 본문에 포함됨: {pic_in_top3}")

# 검증 2: 5월 날씨
print("\n\n" + "=" * 60)
print("Q2: 5월 둘째 주 괌 날씨")
print("=" * 60)
results = vectorstore.similarity_search("5월 둘째 주 괌 날씨", k=3)
for i, doc in enumerate(results):
    cat = doc.metadata.get("category", "?")
    title = doc.metadata.get("title", "?")
    print(f"\n[{i+1}] [{cat}] {title}")
    print(f"    {doc.page_content[:120]}...")

climate_in_top3 = any(
    "Climate" in doc.metadata.get("title", "") or "Climate" in doc.metadata.get("section", "")
    for doc in results
)
print(f"\n→ Climate 청크가 top 3에 포함됨: {climate_in_top3}")


Q1: PIC 리조트 어때?

[1] [리조트] 퍼시픽 아일랜드 클럽 (PIC) - 나무위키
    1. 개요

Pacific Islands Club, PIC

괌 및 북마리아나 제도에 위치한 미국의 복합형 리조트. 흔히 PIC라는 약칭으로 불린다. 괌 및 사이판에 여행하는 관광객들이 가장 많이 찾는 숙박시설이다....

[2] [리조트] 퍼시픽 아일랜드 클럽 (PIC) - 나무위키
    호텔 내 한국인 직원이 상주하고, 외국인 직원 또한 게임등 한국어 패치가 된 분들이 있다.

사이판 최대 번화가인 가라판(Garapan) 시내와 북부와는 거리가 다소 멀긴하나(시내 왕복 무료 서틀버스 운행중) 공항과...

[3] [리조트] 퍼시픽 아일랜드 클럽 (PIC) - 나무위키
    요일별 풀파티 및 별빛 투어 등 투숙객들은 무료로 제공되며 투수객 참여로 팀을 나누어 워터 게임(물총 게임, 줄다리기, 맥주 마시기) 등등 엄청난 이벤트를 제공한다.

식사 프로그램이 다양한데, 크게 골드 카드, 실...

→ 'PIC' 키워드가 top 3 본문에 포함됨: True


Q2: 5월 둘째 주 괌 날씨

[1] [general] 괌 (한국어 위키백과)
    괌에서 가장 높은 산은 람람산, 주물롱망글로산, 볼라노스산, 알마고사산 순이다.

인구

기후

괌의 기후는 열대 우림 기후이지만 낮에는 섭씨 39°C 이상으로 올라가지 않고 밤에도 섭씨 26°C 이하로 내려가지 않...

[2] [general] Guam (English Wikipedia)
    The mean high temperature is 86 °F or 30 °C. The mean low is 76 °F (24.4 °C). Temperatures rarely exceed 90 °F (32.2 °C)...

[3] [general] Guam (English Wikipedia)
    Climate

Guam has a tropical rainforest climate on the Köppen scale (Kö